In [ ]:
# 1. Настройка автоперезагрузки модулей (удобно при разработке)
%load_ext autoreload
%autoreload 2

# 2. Импорт конфигов
import config as cnfg
import config_paths as cnfg_p

# 3. Проверка и создание папок (если их нет)
# При первом запуске он создаст папки data/raw, data/processed и т.д.
cnfg_p.init_structure()

# 4. Библиотечные импорты
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 5. Импорт расчетных модулей
from scripts import import_data as idt
from scripts import data_manager as dm
from scripts import plot_manager as pm
from scripts import pressure_predictor_lite as prm
from scripts import inverse_predictor as iprm


In [ ]:
cnfg_params, flat_params = dm.load_config(cnfg_p.DEFAULT_MODEL_PARAMS_TANK_FILE)

In [ ]:
presets = [0.15, 0.2, 0.3, 0.35, 0.38]
df_logs, df_registry, goal_set = iprm.generate_base_logs_df(
    presets, 
    6, 
    cnfg.CONFIG_GENERATE_TARGET_TANK, 
    flat_params)

In [ ]:
print(goal_set)

In [ ]:
dt = 0.2
time_grid = np.arange(-20.0, 20.0 + dt, dt)
goal_set = np.floor(np.array(goal_set)*0.6)
target = dm.generate_step_profile(
    time_grid, 
    goal_set, 
    6, 
    cnfg.CONFIG_GENERATE_TARGET_TANK['phases'], 
    dm.calculate_step_strategy
)
print(target)


In [ ]:
prediction = prm.analitic_model(time_grid, target, flat_params)
print(prediction)

In [ ]:
p_sim = prm.predict_pressure(
    p_prev=0,
    target_p=5,
    dt=16,
    damping=flat_params['damping'],
    k_gain=flat_params['k_gain'],
    b_gain=flat_params['b_gain']
)

print(p_sim)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_logs['t_relative'], df_logs['goal_pressure_raw'], label='goal_pressure_raw')
plt.plot(time_grid, prediction, label='prediction')
plt.plot(time_grid, target*0.1, label='target')

plt.plot(df_registry['goal_time'], df_registry['goal_pressure_presets'], 'o', label='goal_pressure_presets')
plt.grid(True)
plt.legend()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_logs['t_relative'], df_logs['goal_pressure_raw'], label='goal_pressure_raw')
# plt.plot(df_logs['t_relative'], df_logs['goal_pressure'], label='goal_pressure')
plt.plot(df_logs['t_relative'], df_logs['predicted_pressure'], label='predicted_pressure')
plt.plot(df_registry['goal_time'], df_registry['goal_pressure_presets'], 'o', label='goal_pressure_presets')
plt.grid(True)
plt.legend()